# Local Realtime Inference (Scapy + `.pth` models)

Use this notebook locally with:
- `phase1_ml_model.pth`
- `ft_ae.pth`

Supports PCAP replay and live sniffing.

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scapy.all import rdpcap, sniff, IP, TCP, UDP

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ML_MODEL_PATH = REPO / 'models' / 'phase1_ml' / 'phase1_ml_model.pth'
DL_MODEL_PATH = REPO / 'models' / 'phase2_dl' / 'ft_ae.pth'

USE_PCAP = True
PCAP_PATH = REPO / 'data' / 'sample.pcap'
LIVE_IFACE = 'en0'
LIVE_SECONDS = 20

assert ML_MODEL_PATH.exists(), f'Missing {ML_MODEL_PATH}'
assert DL_MODEL_PATH.exists(), f'Missing {DL_MODEL_PATH}'
if USE_PCAP:
    assert PCAP_PATH.exists(), f'Missing {PCAP_PATH}'

## 1) Load both model artifacts

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, n_features: int, hidden_dims=(128, 64), bottleneck_dim: int = 32):
        super().__init__()
        enc, prev = [], n_features
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.ReLU(inplace=True)]
            prev = h
        enc += [nn.Linear(prev, bottleneck_dim), nn.ReLU(inplace=True)]
        self.encoder = nn.Sequential(*enc)
        dec, prev = [], bottleneck_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(prev, h), nn.ReLU(inplace=True)]
            prev = h
        dec += [nn.Linear(prev, n_features)]
        self.decoder = nn.Sequential(*dec)
    def forward(self, x):
        return self.decoder(self.encoder(x))

with open(ML_MODEL_PATH, 'rb') as f:
    ml_artifact = pickle.load(f)

dl = torch.load(DL_MODEL_PATH, map_location='cpu')
cfg = dl['config']
dl_model = Autoencoder(int(cfg['n_features']), tuple(cfg['hidden_dims']), int(cfg['bottleneck_dim']))
dl_model.load_state_dict(dl['state_dict'])
dl_model.eval()
dl_features = list(dl['feature_names'])
dl_mean = np.asarray(dl['scaler']['mean'], dtype=np.float32)
dl_scale = np.asarray(dl['scaler']['scale'], dtype=np.float32)
dl_thr = float(dl['threshold'])
print('Loaded ML + DL models.')

## 2) Capture packets and build flow dataframe

In [ ]:
def capture_packets(use_pcap: bool):
    if use_pcap:
        return rdpcap(str(PCAP_PATH))
    return sniff(iface=LIVE_IFACE, timeout=LIVE_SECONDS)

def flow_key(pkt):
    if IP not in pkt:
        return None
    ip = pkt[IP]
    proto = int(ip.proto)
    sport = dport = 0
    if TCP in pkt:
        sport, dport = int(pkt[TCP].sport), int(pkt[TCP].dport)
    elif UDP in pkt:
        sport, dport = int(pkt[UDP].sport), int(pkt[UDP].dport)
    return (ip.src, ip.dst, sport, dport, proto)

def packets_to_df(pkts):
    flows = {}
    for pkt in pkts:
        k = flow_key(pkt)
        if k is None: continue
        ts = float(pkt.time); ip = pkt[IP]; plen = len(pkt)
        r = flows.get(k)
        if r is None:
            r = {'start': ts,'end': ts,'pkts':0,'bytes':0,'lens':[],
                 'fwd_pkts':0,'bwd_pkts':0,'fwd_bytes':0,'bwd_bytes':0,
                 'fin':0,'syn':0,'rst':0,'psh':0,'ack':0,'urg':0,'src0':ip.src}
            flows[k] = r
        r['end'] = ts; r['pkts'] += 1; r['bytes'] += plen; r['lens'].append(plen)
        if ip.src == r['src0']:
            r['fwd_pkts'] += 1; r['fwd_bytes'] += plen
        else:
            r['bwd_pkts'] += 1; r['bwd_bytes'] += plen
        if TCP in pkt:
            fl = int(pkt[TCP].flags)
            r['fin'] += 1 if (fl & 0x01) else 0
            r['syn'] += 1 if (fl & 0x02) else 0
            r['rst'] += 1 if (fl & 0x04) else 0
            r['psh'] += 1 if (fl & 0x08) else 0
            r['ack'] += 1 if (fl & 0x10) else 0
            r['urg'] += 1 if (fl & 0x20) else 0
    rows = []
    for (src,dst,sport,dport,proto), r in flows.items():
        dur = max(r['end'] - r['start'], 1e-6)
        lens = np.asarray(r['lens'], dtype=np.float32)
        rows.append({
            'src': src, 'dst': dst, 'sport': sport, 'dport': dport, 'proto': proto,
            'Flow Duration': dur,
            'Total Fwd Packets': float(r['fwd_pkts']),
            'Total Backward Packets': float(r['bwd_pkts']),
            'Total Length of Fwd Packets': float(r['fwd_bytes']),
            'Total Length of Bwd Packets': float(r['bwd_bytes']),
            'Flow Bytes/s': float(r['bytes']/dur),
            'Flow Packets/s': float(r['pkts']/dur),
            'Packet Length Max': float(lens.max()) if len(lens) else 0.0,
            'Packet Length Min': float(lens.min()) if len(lens) else 0.0,
            'Packet Length Mean': float(lens.mean()) if len(lens) else 0.0,
            'Packet Length Std': float(lens.std()) if len(lens) else 0.0,
            'FIN Flag Count': float(r['fin']),
            'SYN Flag Count': float(r['syn']),
            'RST Flag Count': float(r['rst']),
            'PSH Flag Count': float(r['psh']),
            'ACK Flag Count': float(r['ack']),
            'URG Flag Count': float(r['urg']),
            'Average Packet Size': float(lens.mean()) if len(lens) else 0.0,
        })
    return pd.DataFrame(rows)

packets = capture_packets(USE_PCAP)
flow_df = packets_to_df(packets)
print('packets:', len(packets), 'flows:', len(flow_df))
flow_df.head()

## 3) Align features for both models

In [ ]:
def align(df: pd.DataFrame, cols: list[str]) -> np.ndarray:
    out = {}
    for c in cols:
        if c in df.columns:
            out[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0).astype(np.float32)
        else:
            out[c] = pd.Series(np.zeros(len(df), dtype=np.float32))
    return pd.DataFrame(out)[cols].to_numpy(dtype=np.float32)

ml_cols = list(ml_artifact['features'])
X_ml = align(flow_df, ml_cols)
X_ml_s = ml_artifact['scaler'].transform(X_ml)

X_dl = align(flow_df, dl_features)
X_dl_s = (X_dl - dl_mean) / np.where(dl_scale == 0, 1.0, dl_scale)

print('X_ml:', X_ml_s.shape, 'X_dl:', X_dl_s.shape)

## 4) Score and flag anomalies

In [ ]:
iso_score = -ml_artifact['isolation_forest'].score_samples(X_ml_s)
svm_score = -ml_artifact['one_class_svm'].decision_function(X_ml_s)
iso_pred = iso_score >= float(ml_artifact['iso_threshold'])
svm_pred = svm_score >= float(ml_artifact['svm_threshold'])
ml_attack = (iso_pred | svm_pred)

with torch.no_grad():
    xt = torch.from_numpy(X_dl_s.astype(np.float32))
    recon = dl_model(xt)
    dl_err = ((recon - xt) ** 2).mean(dim=1).cpu().numpy()
dl_attack = dl_err >= dl_thr

out = flow_df[['src','dst','sport','dport','proto']].copy()
out['ml_attack'] = ml_attack.astype(int)
out['dl_error'] = dl_err
out['dl_attack'] = dl_attack.astype(int)
out['final_attack'] = ((out['ml_attack'] == 1) | (out['dl_attack'] == 1)).astype(int)
print('flows flagged:', int(out['final_attack'].sum()), '/', len(out))
out.head(30)